# Course–Job Embedding Pipeline

Runs the embedding pipeline on Colab's GPU. All logic lives in `src/embedding/embed.py` — this notebook is just a launcher.

### Setup checklist
1. **GPU**: Runtime → Change runtime type → **T4 GPU**
2. **Data**: The processed CSVs need to be accessible from Google Drive (see below).

### Getting OneDrive data into Google Drive
Colab can’t mount OneDrive natively. The easiest workaround:
1. Open the shared OneDrive folder (`DSA4264_Project_Data/`)
2. Download `processed/jobs/03_jobs_filtered.csv` and `processed/courses/modules_cleaned.csv`
3. Upload them to Google Drive, preserving the folder structure:
   ```
   MyDrive/DSA4264_Project_Data/
     processed/jobs/03_jobs_filtered.csv
     processed/courses/modules_cleaned.csv
     embeddings/    ← outputs will be saved here
   ```
4. After the pipeline runs, download the `embeddings/` folder from Drive and copy it back to OneDrive for teammates.

This is a one-time setup. If the processed data changes, just re-upload the two CSVs.

In [ ]:
# 1. Mount Google Drive & set data path
from google.colab import drive
drive.mount('/content/drive')

# Point this to wherever you put the shared data folder in Drive
DATA_ROOT = "/content/drive/MyDrive/DSA4264_Project_Data"  # <-- CHANGE IF NEEDED

In [ ]:
# 2. Verify data files exist
import os

required = [
    f"{DATA_ROOT}/processed/jobs/03_jobs_filtered.csv",
    f"{DATA_ROOT}/processed/courses/modules_cleaned.csv",
]

all_good = True
for f in required:
    exists = os.path.exists(f)
    print(f"  {'✓' if exists else '✗ MISSING'}  {f}")
    if not exists:
        all_good = False

if not all_good:
    raise FileNotFoundError(
        "Missing data files. See instructions above for uploading "
        "processed CSVs from OneDrive to Google Drive."
    )

# Create embeddings output directory
os.makedirs(f"{DATA_ROOT}/embeddings", exist_ok=True)
print("\n  Data verified. Ready to run.")

In [ ]:
# 3. Clone repo & install dependencies
REPO_URL = "https://github.com/DanDmc/DSA4264_Project.git"  # <-- CHANGE IF NEEDED
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

!git clone -q {REPO_URL}
%cd {REPO_NAME}

!pip install -q sentence-transformers python-dotenv

In [ ]:
# 4. Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# 5. Run the embedding pipeline
#    --data-root bypasses .env (which doesn't exist on Colab)
#    --mode whole_text is the validated baseline
#    Reduce --batch-size to 32 or 16 if you hit GPU OOM

!python -m src.embedding.embed \
    --data-root "{DATA_ROOT}" \
    --mode whole_text \
    --batch-size 64

In [ ]:
# 6. Verify outputs
import os

emb_dir = f"{DATA_ROOT}/embeddings/whole_text"
print(f"Output directory: {emb_dir}\n")

if os.path.exists(emb_dir):
    for fname in sorted(os.listdir(emb_dir)):
        size_mb = os.path.getsize(os.path.join(emb_dir, fname)) / (1024**2)
        print(f"  {fname:50s} {size_mb:>8.2f} MB")
else:
    print("  ERROR: Output directory not found.")

In [ ]:
# 7. Quick sanity check
import numpy as np
import json

emb_dir = f"{DATA_ROOT}/embeddings/whole_text"

mod_emb = np.load(f"{emb_dir}/module_embeddings_bge-large-en-v1.5.npy")
job_emb = np.load(f"{emb_dir}/job_embeddings_bge-large-en-v1.5.npy")

with open(f"{emb_dir}/embedding_config.json") as f:
    config = json.load(f)

print(f"Module embeddings: {mod_emb.shape}")
print(f"Job embeddings:    {job_emb.shape}")
print(f"\nNorm check (should be ~1.0):")
print(f"  Modules: {np.linalg.norm(mod_emb, axis=1).mean():.4f}")
print(f"  Jobs:    {np.linalg.norm(job_emb, axis=1).mean():.4f}")
print(f"\nModel: {config['model']}")
print(f"Created: {config['created_at']}")

In [ ]:
# 8. Download embeddings folder for local use / OneDrive sync
#    Uncomment and run when you're ready to download.

# !zip -r /content/embeddings.zip "{DATA_ROOT}/embeddings/whole_text/"
# from google.colab import files
# files.download('/content/embeddings.zip')